<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/15-text.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 15 — Working With Text

Companion to [the chapter](https://www.ai.biz/books/python-primer/text/).


In [ ]:
import pandas as pd, numpy as np, unicodedata, re


## 1. The problem, made visible

`repr` shows the characters that a plain print hides.


In [ ]:
s = pd.Series(['New York','new york ','  NEW YORK','New York City','New York','new yrok'])
print('unique values:', s.nunique())
print(s.value_counts())
print()
print('with repr, the trailing spaces become visible:')
for v in s.unique(): print(' ', repr(v))


## 2. The normalisation ladder

Check the unique count after each step.


In [ ]:
t = s.copy(); print(f'start                : {t.nunique()}')
t = t.str.strip();                     print(f'after strip          : {t.nunique()}')
t = t.str.lower();                     print(f'after lower          : {t.nunique()}')
t = t.str.replace(r'\s+',' ',regex=True); print(f'after collapse spaces: {t.nunique()}')
print()
print(sorted(t.unique()))


Two lines took six categories to four, with no thought about the data.


## 3. Accent folding

The same visible character can be stored two ways and compare as unequal.


In [ ]:
a = 'caf\u00e9'              # single codepoint e-acute
b = 'cafe\u0301'             # e followed by a combining accent
print(f'{a!r} == {b!r} ->', a == b, ' <- they look identical')

def fold(series):
    return (series.str.normalize('NFKD')
                  .str.encode('ascii', errors='ignore')
                  .str.decode('utf-8'))

print('after folding:', fold(pd.Series([a,b])).tolist())


## 4. Manual fixes belong in a dictionary, not in code branches


In [ ]:
FIXES = {'new york city':'new york', 'nyc':'new york', 'new yrok':'new york'}
key = t.replace(FIXES)
print('final categories:', sorted(key.unique()))
print()
known = {'new york','tokyo'}
print('still unmapped:', sorted(set(key) - known))


A dictionary can be reviewed by a domain expert, tested, and loaded from a file when it grows.


## 5. Extracting with named groups


In [ ]:
orders = pd.Series(['Order #1234 (2026)','Order #5678 (2025)','malformed'])
print(orders.str.extract(r'#(?P<order>\d+) \((?P<year>\d{4})\)'))
print()
print('Unmatched rows become NaN rather than raising.')


## 6. Cleaning with regular expressions


In [ ]:
amounts = pd.Series(['£1,200.50','$3,400','€900.00'])
print(amounts.str.replace(r'[£$€,]','',regex=True).astype(float))
print()
phones = pd.Series(['+44 20 7123 4567','(020) 7123-4567'])
print(phones.str.replace(r'\D','',regex=True))


## 7. Splitting and exploding


In [ ]:
names = pd.Series(['Asha Kumar','Ravi Chandra Reddy'])
print(names.str.split(' ', n=1, expand=True))
print()
tags = pd.DataFrame({'id':[1,2], 'tags':['a,b,c','d,e']})
print(tags.assign(tag=tags.tags.str.split(',')).explode('tag'))


## 8. Fuzzy matching — never apply it silently


In [ ]:
from difflib import get_close_matches
known = ['new york','tokyo','paris','london']

def best(v, options, cutoff=0.8):
    m = get_close_matches(v, options, n=1, cutoff=cutoff)
    return m[0] if m else None

for v in ['new yrok','tokio','nairobi']:
    print(f'{v:<10} -> {best(v, known)}')
print()
print('Produce a review table for a human, then apply the reviewed table.')


## Try it yourself

1. Add `'NEW YORK  '` with two trailing spaces and confirm the ladder still collapses it.
2. Write a regex that extracts the year from both `2026-03-15` and `15/03/2026`.
3. Lower the fuzzy cutoff to 0.6 and see which wrong matches appear.
